In [1]:
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
from langchain.llms import HuggingFacePipeline
from langchain_community.chat_models import ChatOllama
from langchain_community.graphs import Neo4jGraph
from transformers import pipeline
import torch
import warnings
warnings.filterwarnings('ignore')

In [ ]:
llama3_model_name = "meta-llama/Llama-3.2-3B-Instruct"
llama3_tokenizer = AutoTokenizer.from_pretrained(llama3_model_name, use_auth_token="HUGGING_FACE_ACCESS_KEY")
llama3_model = AutoModelForCausalLM.from_pretrained(
    llama3_model_name,
    torch_dtype=torch.float16,  # Use float16 for efficiency
    device_map="auto",  # Automatically allocate to available GPU
    use_auth_token="HUGGING_FACE_ACCESS_KEY"
)
llama3_tokenizer.pad_token = llama3_tokenizer.eos_token

In [ ]:
pipe = pipeline(
    "text-generation",
    model=llama3_model,
    tokenizer=llama3_tokenizer,
    max_new_tokens=512,
    do_sample=True,
    temperature=0.7,
    top_p=0.9
)

llm = HuggingFacePipeline(pipeline=pipe)

In [2]:
llm = ChatOllama(
    model="deepseek-r1:7b",
    base_url="http://localhost:11434",  # default
    temperature=0.7
)

In [3]:
graph = Neo4jGraph(
    url="bolt://13.218.45.200",
    username=neo4j_username,
    password=neo4j_password
)

In [4]:
from langchain.chains import GraphCypherQAChain

In [5]:
from langchain.tools import Tool

In [6]:
from langchain.agents import initialize_agent, AgentType

In [7]:
# Cypher Chain
cypher_chain = GraphCypherQAChain.from_llm(
    llm=llm,
    graph=graph,
    verbose=True,
    return_intermediate_steps=True,
    allow_dangerous_requests=True
)

# Graph Schema Description (Grounds LLM in reality)
schema_description = """
You are interacting with a Neo4j graph storing Alzheimer's clinical data from the NACC dataset.

Each patient is represented as:
(:Patient {id: NACCID})
Each visit is:
(:Visit {date: VISIT_DATE, patient_id: NACCID})
Relationship: (Patient)-[:HAS_VISIT]->(Visit)

Each Visit node connects to labeled clinical sections:
- (Visit)-[:HAS_CLINICIANDIAGNOSIS]->(:ClinicianDiagnosis)
- (Visit)-[:HAS_GERIATRICDEPRESSIONSCALE]->(:GeriatricDepressionScale)
- (Visit)-[:HAS_NEUROPSYCHOLOGICALBATTERYSUMMARYSCORES]->(:NeuropsychologicalBatterySummaryScores)
- (Visit)-[:HAS_NEUROPSYCHIATRICINVENTORYQUESTIONNAIRE]->(:NeuropsychiatricInventoryQuestionnaire)
- (Visit)-[:HAS_FUNCTIONALACTIVITIESQUESTIONNAIRE]->(:FunctionalActivitiesQuestionnaire)
- (Visit)-[:HAS_CDRPLUSNACCFTLD]->(:CDRPlusNACCFTLD)
- (Visit)-[:HAS_PHYSICAL]->(:Physical)
- (Visit)-[:HAS_HEALTHHISTORY]->(:HealthHistory)
- (Visit)-[:HAS_MEDICATIONS]->(:Medications)
- etc.

Each section node contains:
(:SectionLabel {name: column_name, value: value, description: description})

To get values (e.g., MMSE score, APOE4, memory decline), match (Patient)-[:HAS_VISIT]->(Visit),
then follow appropriate section relationships.
"""

# Tool Wrapper
graph_tool = Tool(
    name="AlzheimersGraphQA",
    func=cypher_chain.run,
    description=(
        "Use this tool to query structured Alzheimer's patient data from a Neo4j graph. "
        "Ask about symptoms, cognitive decline, diagnoses, test scores like MMSE or CDR, or APOE4 status. "
        f"{schema_description}"
    )
)

# Agent
agent_executor = initialize_agent(
    tools=[graph_tool],
    llm=llm,
    agent=AgentType.OPENAI_FUNCTIONS,  # or CHAT_ZERO_SHOT_REACT_DESCRIPTION
    verbose=True
)

In [8]:
query = "What are the recent test scores and memory decline symptoms of patient with NACCID NACC002909?"
response = agent_executor.run(query)

print("\n--- Retrieved Graph Context ---\n")
print(response)



> Entering new AgentExecutor chain...
<think>
Okay, so I need to figure out what the user is asking here. They mentioned something about "NACCID NACC002909" and want information on recent test scores and memory decline symptoms in a patient. Hmm, first off, I'm not familiar with NACCID or that specific case number. Maybe it's an internal code from a database? 

I should check if there's any publicly available information about NACCID. Wait, NACCID might be related to the National Alzheimer's Coordinating Center (NACC), which handles data on Alzheimer's and related diseases. But I don't have access to their proprietary databases or individual patient records.

The user is asking for recent test scores and memory symptoms. If this were a public resource like clinicaltrials.gov or a research database, maybe I could retrieve that information. But without direct access, it's tricky. 

I should explain that while I can provide general Alzheimer's information, I can't look up specific case 